# KNN / WKNN trên UJIIndoorLoc

Chọn một hoặc nhiều model KNN/WKNN; notebook in thông số và vẽ bảng, CDF, sai số theo tòa, thời gian và ma trận nhầm lẫn **tầng**. Repo private: dùng Colab Secret `GITHUB_TOKEN` có quyền **Contents: Read**, hoặc upload ZIP của nhánh. Không ghi token vào notebook hay output. Để báo cáo kết quả chính thức, chọn full; smoke chỉ kiểm tra luồng.


In [ ]:
# 1. Tải repo private. Sau khi có commit mới, chạy lại ô này.
import io
import shutil
import subprocess
import urllib.error
import urllib.request
import zipfile
from pathlib import Path
from google.colab import files

REPO = "VietUniverse/ips-wifi-research"
BRANCH = "feature/knn-wknn"  # sau khi PR được merge có thể đổi thành main
ZIP_PATH = None  # hoặc đường dẫn ZIP đã upload, ví dụ "/content/ips-wifi-research.zip"

repo_zip = Path(ZIP_PATH).read_bytes() if ZIP_PATH else None
if repo_zip is None:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    if token:
        try:
            request = urllib.request.Request(
                f"https://api.github.com/repos/{REPO}/zipball/{BRANCH}",
                headers={"Authorization": f"Bearer {str(token).strip()}",
                         "Accept": "application/vnd.github+json"})
            with urllib.request.urlopen(request, timeout=120) as response:
                repo_zip = response.read()
        except urllib.error.HTTPError as error:
            print(f"GitHub trả HTTP {error.code}. Kiểm tra token có quyền Contents: Read,"
                  " đã chọn đúng repo, bật quyền truy cập Secret cho notebook và BRANCH tồn tại.")
        except urllib.error.URLError:
            print("Không kết nối được GitHub API. Có thể dùng ZIP của repo để tiếp tục.")
    token = None
if repo_zip is None:
    print("Upload ZIP của đúng nhánh repo từ GitHub (Code → Download ZIP).")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Hãy upload đúng một file ZIP")
    repo_zip = next(iter(uploaded.values()))

root = Path("/content/knn_repo_source")
if root.exists():
    shutil.rmtree(root)
root.mkdir()
with zipfile.ZipFile(io.BytesIO(repo_zip)) as archive:
    archive.extractall(root)
projects = list(root.rglob("pyproject.toml"))
if len(projects) != 1:
    raise RuntimeError(f"ZIP cần đúng một pyproject.toml; tìm thấy {len(projects)}")
project = projects[0].parent
if not (project / "src/ips/neighbors").is_dir():
    raise RuntimeError("Nhánh/ZIP này chưa có KNN. Chọn BRANCH=feature/knn-wknn hoặc ZIP của nhánh KNN.")
print("Project:", project)
subprocess.run(["python", "-m", "pip", "install", "-e", str(project)], check=True)
subprocess.run(["python", "-m", "ips.experiments.knn_cli", "--list-options"], check=True)


In [ ]:
# 2. Tải UJIIndoorLoc từ UCI nếu chưa có hai CSV.
import urllib.request

data_dir = Path("/content/data/uji")
data_dir.mkdir(parents=True, exist_ok=True)
if not all((data_dir / name).exists() for name in ("trainingData.csv", "validationData.csv")):
    zip_path = Path("/content/ujiindoorloc.zip")
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/310/ujiindoorloc.zip", zip_path)
    extracted = Path("/content/uji_source")
    extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(extracted)
    for name in ("trainingData.csv", "validationData.csv"):
        matches = list(extracted.rglob(name))
        if len(matches) != 1:
            raise RuntimeError(f"Không tìm thấy duy nhất {name} trong ZIP UJI")
        shutil.copy2(matches[0], data_dir / name)
print("UJI data:", [(p.name, p.stat().st_size) for p in data_dir.glob("*.csv")])


In [ ]:
# 3. Chọn một hoặc nhiều model; smoke kiểm tra luồng, full dùng toàn bộ dữ liệu.
from datetime import datetime, timezone
import json
from ips.neighbors import available_models
from IPython.display import display

print('Model khả dụng:', sorted(available_models()))
MODELS = sorted(available_models())  # Ví dụ chỉ chạy một: ['knn']
CONFIG = "configs/knn_smoke.json"  # đổi thành configs/knn_full.json khi sẵn sàng
settings = json.loads((project / CONFIG).read_text(encoding='utf-8'))
if not MODELS or any(m not in available_models() for m in MODELS):
    raise ValueError('MODELS phải là các model trong danh sách khả dụng')
settings['models'] = MODELS
selected_config = Path('/content/selected_knn_config.json')
selected_config.write_text(json.dumps(settings, indent=2), encoding='utf-8')
print('Cấu hình:', json.dumps(settings, ensure_ascii=False, indent=2))
print('Số dòng dữ liệu:', [(p.name, sum(1 for _ in p.open()) - 1) for p in data_dir.glob('*.csv')])
run_dir = Path("/content/runs") / datetime.now(timezone.utc).strftime("knn_%Y%m%d_%H%M%S_%f")
subprocess.run(["python", "-m", "ips.experiments.knn_cli", "--config", str(selected_config),
                "--data-dir", str(data_dir), "--output-dir", str(run_dir)], check=True)
print((run_dir / "summary.json").read_text(encoding="utf-8"))
from ips.experiments.visualize import render_run
report_table, report_figures = render_run(run_dir)
print('Ảnh báo cáo:', [str(p) for p in report_figures])


# Mở rộng: quét số WAP (m) theo luồng Lần 6.5

Phần này chạy **sau** ô 3 và **trước** ô tải kết quả; các ô 1–4 phía trên giữ nguyên. Luồng giống Lần 6.5: xếp hạng WAP chỉ bằng dữ liệu train (**RF** và **Độ phủ**), giữ top-m WAP (m = 200 → 300, bước 10, cộng baseline dùng toàn bộ WAP), tìm 20 láng giềng một lần cho mỗi m rồi quét k = 1 → 20 cho kNN, WKNN (1/d), WKNN (1/d²). Sau đó in một bảng cho mỗi m, bảng tổng hợp có kiểm định Wilcoxon (ngưỡng Bonferroni), tự chọn m, vẽ biểu đồ và lưu CSV/PNG vào thư mục `m_sweep` trong thư mục kết quả của ô 3, nên ô tải kết quả sẽ tự gói chúng vào ZIP.

Các hàm công thức và model dùng lại của repo (`Formulas`, `build_model`: biểu diễn powed, khoảng cách Sørensen, vote tòa → tầng, trọng số, vị trí). **Giao thức khác ô 3:** ô 3 chạy từng tòa theo giao thức repo (biết sẵn tòa, k chọn trên validation nội bộ), còn phần này theo Lần 6.5: tìm láng giềng trên toàn bộ train, vote tòa → tầng, đánh giá trên `validationData.csv` và tính sai số trên các mẫu đúng cả tòa lẫn tầng. Vì vậy số liệu ở đây **không so trực tiếp** với bảng của ô 3.

⚠️ UJIIndoorLoc chỉ có train và validation, nên chọn m và k trên validation là chọn trên tập kiểm tra (hơi lạc quan). Đặt `RUN_M_SWEEP = False` ở ô kế tiếp để bỏ qua toàn bộ phần này.

In [ ]:
# 3b. Cấu hình và nạp dữ liệu (dùng lại data_dir, project, settings, run_dir của các ô trước)
RUN_M_SWEEP = True            # False để bỏ qua toàn bộ phần quét m
if RUN_M_SWEEP:

    from ips.data.uji import load_csvs, AP_COLUMNS
    from ips.neighbors import build_model
    from ips.neighbors.formulas import Formulas
    import numpy as np, pandas as pd, time, os
    import matplotlib.pyplot as plt

    ABS_M = list(range(200, 301, 10))       # 200, 210, ..., 300; baseline m = N_WAP tự thêm bên dưới
    N_REP = 3                               # số lần đo thời gian predict
    K_MAX = int(settings.get('k_max', 20))
    RANDOM_STATE, N_TREES, W_CLS = 42, 100, 0.5
    RANK_FILE = '/content/L5_wap_ranking.npz'
    SR_TOL, SR_SMALL, ERR_SMALL = 0.5, 1.0, 5.0
    sweep_dir = run_dir / 'm_sweep'; sweep_dir.mkdir(parents=True, exist_ok=True)
    PNGS = []
    def save_fig(fig, fname):
        path = sweep_dir / fname
        fig.tight_layout(); fig.savefig(path, dpi=150); PNGS.append(path); plt.show()

    train_df, val_df = load_csvs(data_dir)
    wap_cols = list(AP_COLUMNS)
    raw_tr, raw_val = train_df[wap_cols].to_numpy(), val_df[wap_cols].to_numpy()
    N_WAP = len(wap_cols)
    X_train, X_val = Formulas.to_powed(raw_tr), Formulas.to_powed(raw_val)       # powed; WAP không bắt được = 0
    tb, tf = train_df.BUILDINGID.to_numpy(int), train_df.FLOOR.to_numpy(int)
    tx, ty = train_df.LONGITUDE.to_numpy(), train_df.LATITUDE.to_numpy()
    vb, vf = val_df.BUILDINGID.to_numpy(int), val_df.FLOOR.to_numpy(int)
    vx, vy = val_df.LONGITUDE.to_numpy(), val_df.LATITUDE.to_numpy()
    NB, NF = int(max(tb.max(), vb.max())) + 1, int(max(tf.max(), vf.max())) + 1
    N_UNSEEN = int((~(raw_tr != 100).any(axis=0)).sum())
    print(f'N_WAP = {N_WAP} | train = {len(X_train)} | validation = {len(X_val)} | WAP chưa từng thấy trong train: {N_UNSEEN}')

In [ ]:
# 3c. Model (lấy từ repo), hàm dự đoán toàn cục và đánh giá theo Lần 6.5
if RUN_M_SWEEP:
    METHODS = ('knn', 'wknn_inv_d', 'wknn_inv_d2')
    MODELS = {me: build_model(me).fit(tf, tx, ty, NF) for me in METHODS}      # fit chỉ ghi nhớ CSDL tham chiếu
    NAME = {me: MODELS[me].display_name for me in METHODS}
    COL = {me: MODELS[me].color for me in METHODS}

    def predict(indices, dists, k, method, model_map=MODELS, ref=(tb, tx, ty)):
        # vote tòa -> tầng trên toàn bộ train, rồi trọng số của model repo và gộp tọa độ
        rb, rx, ry = ref
        nb, d = indices[:, :k], dists[:, :k]
        pb, pf, mask = Formulas.vote_building_floor(rb[nb], model_map[method].rf[nb], NB, NF)
        w, n0 = model_map[method].weights(d, mask)
        px, py = Formulas.weighted_position(w, rx[nb], ry[nb])
        return pb, pf, px, py, n0

    def evaluate(pb, pf, px, py):
        ok = (pb == vb) & (pf == vf); e_all = Formulas.position_error(px, py, vx, vy); e = e_all[ok]
        s = Formulas.error_summary(e)
        return dict(success_rate=ok.mean() * 100, mean_error=s['mean'], median=s['median'], rmse=s['rmse'], p90=s['p90'],
                    le1=s['le1'], le3=s['le3'], le5=s['le5']), ok, e_all

    # kiểm tra tay: d = (2, 4, 8), P1 = (0,0), P2 = (10,0), P3 = (0,10)
    hand = {me: build_model(me).fit(np.zeros(3, int), np.array([0., 10, 0]), np.array([0., 0, 10]), 1) for me in METHODS}
    i3, d3 = np.array([[0, 1, 2]]), np.array([[2., 4, 8]])
    xy = lambda r: (float(r[2][0]), float(r[3][0]))
    href = (np.zeros(3, int), np.array([0., 10, 0]), np.array([0., 0, 10]))
    assert np.allclose(xy(predict(i3, d3, 3, 'wknn_inv_d', hand, href)), (20/7, 10/7))
    assert np.allclose(xy(predict(i3, d3, 3, 'wknn_inv_d2', hand, href)), (40/21, 10/21))
    assert np.allclose(xy(predict(i3, d3, 3, 'knn', hand, href)), (10/3, 10/3))
    print('Kiểm tra tay OK')

## Hai cách xếp hạng WAP (chỉ dùng train)
- **`rf`**: Random Forest (phân loại tòa + tầng, hồi quy x, y; importance trung bình 50/50). Có cache `/content/L5_wap_ranking.npz` (dùng lại file của Lần 5 nếu đã có, hoặc huấn luyện mới; xóa file nếu đổi dữ liệu).
- **`coverage`** (độ phủ): WAP nào được bắt ở **nhiều mẫu train nhất** thì xếp trước. Rẻ, và hạn chế mẫu toàn 0 khi m nhỏ. Dùng để so với RF: nếu RF không hơn độ phủ thì chi phí huấn luyện RF không đáng.

In [ ]:
# 3d. Xếp hạng WAP: RF và Độ phủ (chỉ dùng train)
if RUN_M_SWEEP:
    from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

    if os.path.exists(RANK_FILE):
        z = np.load(RANK_FILE)
        imp_cls, imp_reg, T_RF = z['imp_cls'], z['imp_reg'], float(z['t_rf'])
        assert len(imp_cls) == N_WAP, 'File xếp hạng cũ không khớp số WAP, hãy xóa file rồi chạy lại.'
        print(f'Đã nạp xếp hạng RF từ {RANK_FILE} (T_RF = {T_RF:.1f}s)')
    else:
        y_cls = tb * NF + tf; Y_reg = np.column_stack([tx, ty])
        t0 = time.perf_counter()
        rf_cls = RandomForestClassifier(n_estimators=N_TREES, max_features='sqrt', random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_cls)
        rf_reg = RandomForestRegressor(n_estimators=N_TREES, max_features='sqrt', random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, Y_reg)
        T_RF = time.perf_counter() - t0
        imp_cls, imp_reg = rf_cls.feature_importances_, rf_reg.feature_importances_
        np.savez(RANK_FILE, imp_cls=imp_cls, imp_reg=imp_reg, t_rf=T_RF)
        print(f'Đã huấn luyện RF: T_RF = {T_RF:.1f}s')
    imp = W_CLS * imp_cls / imp_cls.sum() + (1 - W_CLS) * imp_reg / imp_reg.sum()
    order_rf = np.argsort(-imp, kind='stable')

    t0 = time.perf_counter()
    cov = (raw_tr != 100).sum(0)                                   # số mẫu train bắt được mỗi WAP
    order_cov = np.argsort(-cov, kind='stable')
    T_COV = time.perf_counter() - t0

    RANKINGS = {'rf': order_rf, 'coverage': order_cov}            # chỉ giữ {'rf': order_rf} nếu muốn chạy nhanh
    RANK_T = {'rf': T_RF, 'coverage': T_COV}
    RNAME = {'rf': 'RF', 'coverage': 'Độ phủ'}
    overlap = {m: len(set(order_rf[:m]) & set(order_cov[:m])) for m in (10, 50, 100, 200)}
    print('Số WAP trùng nhau giữa 2 cách xếp hạng ở top-m:', overlap)

## Quét hàng loạt m × k × phương pháp × cách xếp hạng
Với mỗi (cách xếp hạng, m): chọn top-m cột → tìm 20 láng giềng **một lần** (khoảng cách Sørensen của repo) → quét k = 1 → 20 cho 3 phương pháp. `t_search_s` dùng chung cho 3 phương pháp; `total_with_rank_s` = tổng + thời gian tạo xếp hạng (chỉ khi m < `N_WAP`). Muốn chạy nhanh hơn: bỏ bớt mức trong `ABS_M` hoặc để `RANKINGS` chỉ còn RF.

In [ ]:
# 3e. Quét m × k × phương pháp × cách xếp hạng
if RUN_M_SWEEP:
    M_LIST = sorted({m for m in ABS_M if 1 <= m < N_WAP} | {N_WAP})
    print('Các mức m:', M_LIST, '| N_WAP =', N_WAP)

    rows, RES, ZERO = [], {}, {}
    for rk, order_ in RANKINGS.items():
        for m in M_LIST:
            cols_m = order_[:m]
            t0 = time.perf_counter()
            Xt = np.ascontiguousarray(X_train[:, cols_m]); Xv = np.ascontiguousarray(X_val[:, cols_m])
            t_sel = time.perf_counter() - t0
            ZERO[(rk, m)] = (int((Xt.sum(1) == 0).sum()), int((Xv.sum(1) == 0).sum()))
            t0 = time.perf_counter(); dist_m, idx_m = Formulas.sorensen_search(Xv, Xt, K_MAX); t_srch = time.perf_counter() - t0
            t_rank = RANK_T[rk] if m < N_WAP else 0.0
            for me in METHODS:
                for k in range(1, K_MAX + 1):
                    ts = []
                    for _ in range(N_REP):
                        t0 = time.perf_counter(); out = predict(idx_m, dist_m, k, me); ts.append(time.perf_counter() - t0)
                    pb, pf, px, py, n0 = out
                    r, ok, e_all = evaluate(pb, pf, px, py)
                    tot = t_sel + t_srch + min(ts)
                    rows.append(dict(rk=rk, m=m, pct_wap=100 * m / N_WAP, method=me, k=k, n_query_d0=n0, **r,
                                     t_select_s=t_sel, t_search_s=t_srch, t_predict_s=min(ts), total_s=tot,
                                     t_rank_s=t_rank, total_with_rank_s=tot + t_rank,
                                     zero_train=ZERO[(rk, m)][0], zero_val=ZERO[(rk, m)][1]))
                    RES[(rk, m, me, k)] = (ok, e_all)
        print(f'[{RNAME[rk]}] xong {len(M_LIST)} mức m')

    sweep = pd.DataFrame(rows)
    SW = sweep.set_index(['rk', 'm', 'method', 'k']).sort_index()
    BEST = {}
    for rk in RANKINGS:
        for m in M_LIST:
            for me in METHODS:
                s = sweep[(sweep.rk == rk) & (sweep.m == m) & (sweep.method == me)].sort_values(['success_rate', 'mean_error'], ascending=[False, True])
                BEST[(rk, m, me)] = int(s.iloc[0].k)
    BASE_RK = 'rf' if 'rf' in RANKINGS else next(iter(RANKINGS))     # baseline m = N_WAP giống nhau ở mọi cách xếp hạng
    K_FIX = BEST[(BASE_RK, N_WAP, 'wknn_inv_d2')]                    # k chung = k tốt nhất của WKNN (1/d²) ở baseline
    def get(rk, m, me, k=None):
        return SW.loc[(rk, m, me, BEST[(rk, m, me)] if k is None else k)]
    print(f'Đã chạy {len(sweep)} lần đánh giá trên {len(vx)} mẫu. K_FIX = {K_FIX} (k tốt nhất của WKNN 1/d² ở baseline m = {N_WAP})')

## Mỗi m một bảng đánh giá
Mỗi bảng ứng với **một (cách xếp hạng, m)**, gồm 3 phương pháp × 2 cách chọn k (k tốt nhất riêng và cùng `K_FIX`). `d_SR` và `d_err` là **chênh so với baseline** (m = `N_WAP`, cùng phương pháp và cùng cách chọn k): `d_SR` dương = tốt hơn, `d_err` **âm = chính xác hơn**. `% mẫu val toàn 0` là tỉ lệ mẫu kiểm tra không bắt được WAP nào trong top-m. Đặt `SHOW_RANKINGS`/`SHOW_M` để in bớt (mọi bảng vẫn được lưu vào CSV).

In [ ]:
# 3f. Bảng đánh giá cho từng m
if RUN_M_SWEEP:
    SHOW_RANKINGS = list(RANKINGS)      # ví dụ ['rf'] để chỉ in bảng của RF
    SHOW_M = M_LIST                     # ví dụ [200, 250, 300] để chỉ in vài mức

    def m_table(rk, m):
        out = []
        for me in METHODS:
            for tag, kk in (('k tốt nhất riêng', None), (f'cùng k = {K_FIX}', K_FIX)):
                a = get(rk, m, me, kk); b = get(BASE_RK, N_WAP, me, kk)
                out.append({'Phương pháp': NAME[me], 'Cách chọn k': tag, 'k': int(a.name[3]), 'Success rate (%)': a.success_rate,
                            'd_SR (điểm)': a.success_rate - b.success_rate, 'Mean (m)': a.mean_error, 'Median (m)': a['median'],
                            'RMSE (m)': a.rmse, 'P90 (m)': a.p90, '≤1m (%)': a.le1, '≤3m (%)': a.le3, '≤5m (%)': a.le5,
                            'd_err (m)': a.mean_error - b.mean_error, 'd_err (%)': 100 * (a.mean_error / b.mean_error - 1),
                            'Tổng (s)': a.total_s, 'Tổng có xếp hạng (s)': a.total_with_rank_s,
                            '% mẫu val toàn 0': 100 * ZERO[(rk, m)][1] / len(X_val)})
        return pd.DataFrame(out)

    TABLES = {}
    for rk in RANKINGS:
        for m in M_LIST:
            TABLES[(rk, m)] = m_table(rk, m)
            if rk in SHOW_RANKINGS and m in SHOW_M:
                print(f'\n===== Xếp hạng: {RNAME[rk]} | m = {m} ({100 * m / N_WAP:.1f}% N_WAP) =====')
                display(TABLES[(rk, m)].round(3))

## Bảng tổng hợp mọi m (cùng k = `K_FIX`) và kiểm định
Mỗi dòng là một (cách xếp hạng, m, phương pháp). Wilcoxon cặp so sai số với baseline trên các mẫu **đúng tòa + tầng ở cả hai cấu hình**; ngưỡng Bonferroni = 0,05 / (số mức m < `N_WAP` đã thử). Cột `improved` = True khi **sai số nhỏ hơn có ý nghĩa thống kê** và success rate không giảm quá `SR_TOL` điểm.

In [ ]:
# 3g. Tổng hợp mọi m + kiểm định Wilcoxon với baseline
if RUN_M_SWEEP:
    from scipy.stats import wilcoxon
    ALPHA = 0.05 / max(len(M_LIST) - 1, 1)
    recs = []
    for rk in RANKINGS:
        for m in M_LIST:
            for me in METHODS:
                a = get(rk, m, me, K_FIX); b = get(BASE_RK, N_WAP, me, K_FIX)
                ok1, e1 = RES[(rk, m, me, K_FIX)]; ok0, e0 = RES[(BASE_RK, N_WAP, me, K_FIX)]
                both = ok1 & ok0; dl = e1[both] - e0[both]; pv = np.nan
                if m < N_WAP:
                    try: pv = wilcoxon(e1[both], e0[both]).pvalue
                    except ValueError: pass
                recs.append(dict(rk=rk, m=m, pct_wap=a.pct_wap, method=me, success_rate=a.success_rate, d_SR=a.success_rate - b.success_rate,
                                 mean_error=a.mean_error, d_err=a.mean_error - b.mean_error, d_err_pct=100 * (a.mean_error / b.mean_error - 1),
                                 n_paired=int(both.sum()), d_err_paired=dl.mean(), p_value=pv,
                                 zero_val_pct=100 * ZERO[(rk, m)][1] / len(X_val), total_s=a.total_s,
                                 total_with_rank_s=a.total_with_rank_s, speedup=b.total_s / a.total_s))
    ALL = pd.DataFrame(recs); ALL['p_value'] = ALL.p_value.astype(float)
    ALL['improved'] = (ALL.d_err_paired < 0) & (ALL.p_value < ALPHA) & (ALL.d_SR >= -SR_TOL)

    print(f'Ngưỡng Bonferroni = {ALPHA:.5f} | SR_TOL = {SR_TOL} điểm')
    t_sr = ALL.pivot_table(index='m', columns=['rk', 'method'], values='success_rate')
    t_er = ALL.pivot_table(index='m', columns=['rk', 'method'], values='mean_error')
    print('Success rate (%) theo m:'); display(t_sr.round(2))
    print('Sai số trung bình (m) theo m:'); display(t_er.round(3))
    print('Chi tiết đầy đủ (kèm d_SR, d_err, p-value, mẫu toàn 0, thời gian):')
    display(ALL.round(4))

## Chọn số lượng đặc trưng m
Với mỗi (cách xếp hạng, phương pháp) in ra: (1) **m cho sai số thấp nhất** (trong các m mà success rate giảm không quá `SR_TOL` điểm) và nó có **cải thiện có ý nghĩa thống kê** so với dùng toàn bộ WAP hay không; (2) **m nhỏ nhất** mà success rate giảm ≤ `SR_SMALL` điểm và sai số tăng ≤ `ERR_SMALL`%. Cuối cùng là cấu hình tốt nhất tổng thể.

In [ ]:
# 3h. Chọn m
if RUN_M_SWEEP:
    sel = []
    print('=' * 90)
    for rk in RANKINGS:
        for me in METHODS:
            s = ALL[(ALL.rk == rk) & (ALL.method == me) & (ALL.m < N_WAP)]
            base_err = float(get(BASE_RK, N_WAP, me, K_FIX).mean_error)
            ok_sr = s[s.d_SR >= -SR_TOL]
            line = f'[{RNAME[rk]} | {NAME[me]}] baseline (m={N_WAP}): {base_err:.3f} m. '
            rec = dict(rk=rk, method=me, baseline_err=base_err)
            if len(ok_sr):
                r = ok_sr.loc[ok_sr.mean_error.idxmin()]
                tag = 'CẢI THIỆN có ý nghĩa thống kê' if r.improved else ('thấp hơn nhưng KHÔNG có ý nghĩa thống kê' if r.d_err < 0 else 'KHÔNG tốt hơn baseline')
                line += f'm sai số thấp nhất = {int(r.m)} ({r.pct_wap:.0f}% N_WAP): {r.mean_error:.3f} m ({r.d_err:+.3f} m, {r.d_err_pct:+.1f}%), SR {r.success_rate:.2f}% ({r.d_SR:+.2f}), p = {r.p_value:.3g} → {tag}. '
                rec.update(m_best=int(r.m), best_err=r.mean_error, best_d_err_pct=r.d_err_pct, best_improved=bool(r.improved))
            else:
                line += 'không m nào giữ được success rate. '
            g = s[(s.d_SR >= -SR_SMALL) & (s.d_err_pct <= ERR_SMALL)]
            if len(g):
                r2 = g.loc[g.m.idxmin()]
                line += f'm nhỏ nhất gần như không mất chính xác = {int(r2.m)} ({r2.pct_wap:.0f}% N_WAP, nhanh hơn {r2.speedup:.1f} lần không tính xếp hạng).'
                rec.update(m_small=int(r2.m), m_small_pct=r2.pct_wap)
            else:
                line += f'không m nào có SR giảm ≤ {SR_SMALL} điểm và sai số tăng ≤ {ERR_SMALL}%.'
            print(line); sel.append(rec)
    print('=' * 90)
    SEL = pd.DataFrame(sel); display(SEL.round(3))

    imp_df = ALL[ALL.improved]
    if len(imp_df):
        r = imp_df.loc[imp_df.mean_error.idxmin()]
        print(f'→ CẤU HÌNH TỐT NHẤT: xếp hạng {RNAME[r.rk]}, {NAME[r.method]}, m = {int(r.m)} ({r.pct_wap:.0f}% N_WAP): {r.mean_error:.3f} m so với baseline {r.mean_error - r.d_err:.3f} m ({r.d_err_pct:+.1f}%), p = {r.p_value:.3g}.')
        print(f'  Có {len(imp_df)} cấu hình (xếp hạng × m × phương pháp) cải thiện có ý nghĩa thống kê; xem bảng dưới.')
        display(imp_df.round(4))
    else:
        print('→ KHÔNG có m nào cải thiện có ý nghĩa thống kê so với dùng toàn bộ WAP. Giữ toàn bộ WAP nếu ưu tiên độ chính xác;')
        print('  nếu ưu tiên gọn/nhanh, dùng m nhỏ nhất ở cột m_small ở trên (gần như không mất chính xác).')
    print('Lưu ý: m và k chọn trên validation, thử nhiều m nên kết quả hơi lạc quan; hãy đọc xu hướng cả đường cong (biểu đồ bên dưới).')

## Biểu đồ và lưu kết quả
Hàng 1 là sai số trung bình theo m (đường chấm cùng màu = baseline của phương pháp đó, dùng toàn bộ WAP; điểm nằm **dưới** đường là tốt hơn baseline), hàng 2 là success rate theo m, hàng 3 là tỉ lệ mẫu validation toàn 0. Màu = phương pháp, nét liền = RF, nét đứt = Độ phủ. Chỉ vẽ các m < `N_WAP`. File CSV/PNG được lưu vào `m_sweep` trong thư mục kết quả của ô 3.

In [ ]:
# 3i. Biểu đồ và lưu kết quả vào run_dir/m_sweep (ô tải kết quả sẽ gói chung vào ZIP)
if RUN_M_SWEEP:
    LS = {'rf': '-', 'coverage': '--'}
    fig, axes = plt.subplots(3, 1, figsize=(10, 13), sharex=True)
    base_e = {me: float(get(BASE_RK, N_WAP, me, K_FIX).mean_error) for me in METHODS}
    base_sr = float(get(BASE_RK, N_WAP, 'knn', K_FIX).success_rate)
    P = ALL[ALL.m < N_WAP]
    for rk in RANKINGS:
        for me in METHODS:
            s = P[(P.rk == rk) & (P.method == me)].sort_values('m')
            axes[0].plot(s.m, s.mean_error, 'o' + LS[rk], color=COL[me], ms=3, label=f'{NAME[me]} - {RNAME[rk]}')
            axes[1].plot(s.m, s.success_rate, 'o' + LS[rk], color=COL[me], ms=3)
        s = P[(P.rk == rk) & (P.method == 'knn')].sort_values('m')
        axes[2].plot(s.m, s.zero_val_pct, 'o' + LS[rk], color='#8172B3', ms=3, label=RNAME[rk])
    for me in METHODS:
        axes[0].axhline(base_e[me], color=COL[me], ls=':', lw=.9)
    axes[1].axhline(base_sr, color='k', ls=':', lw=.9)
    axes[0].set_ylabel('Sai số trung bình (m)'); axes[0].set_title(f'Sai số vị trí theo m (cùng k = {K_FIX}; đường chấm = baseline m = {N_WAP})'); axes[0].legend(fontsize=7, ncol=2)
    axes[1].set_ylabel('Success rate (%)'); axes[1].set_title('Success rate theo m (đường chấm = baseline)')
    axes[2].set_ylabel('% mẫu validation toàn 0'); axes[2].set_title('Mẫu không bắt được WAP nào trong top-m'); axes[2].legend()
    axes[2].set_xlabel('m (số WAP giữ lại)')
    for a in axes: a.grid(alpha=.3)
    save_fig(fig, 'm_sweep.png')

    sweep.to_csv(sweep_dir / 'khao_sat_m_k.csv', index=False)
    ALL.to_csv(sweep_dir / 'tong_hop_theo_m.csv', index=False)
    SEL.to_csv(sweep_dir / 'chon_m.csv', index=False)
    pd.concat([t.assign(xep_hang=RNAME[rk], m=m) for (rk, m), t in TABLES.items()]).to_csv(sweep_dir / 'bang_tung_m.csv', index=False)
    print('Đã lưu vào', sweep_dir, ':', ', '.join(sorted(p.name for p in sweep_dir.iterdir())))

In [ ]:
# 4. Tải kết quả về máy.
archive_path = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
print("Result ZIP:", archive_path)
files.download(archive_path)
